# 13. Bagging & Random Forest

Compare two ensemble methods built on decision trees:
1. **Bagging** — bootstrap samples + vote (same features available at each split)
2. **Random Forest** — bagging + random feature subsets at each split

Baseline single tree is already in `08_decision_tree.ipynb` (`max_depth=15`).
Same preprocessing: `scale=False`.

In [1]:
import sys
sys.path.append('..')

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult('train'))
X_test, y_test = split_features_target(load_adult('test'))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
def run_and_save(name, model, extra):
    pipe = make_model_pipeline(model, scale=False)
    cv = cross_val_scores(pipe, X, y)
    print("CV:", cv)

    pipe2 = make_model_pipeline(clone(model), scale=False)
    test = evaluate_on_test(pipe2, X, y, X_test, y_test)
    print("Test:", test['metrics'])
    print("Confusion matrix:", test['confusion_matrix'])

    path = save_results(name, cv, test, extra)
    print("Saved to", path)



In [3]:
# Decision Tree Bagging

base = DecisionTreeClassifier(max_depth=15, random_state=42)
bag = BaggingClassifier(
    estimator=base,
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

run_and_save(
    "bagging_tree", 
    bag,
    extra = {"base": "DecisionTreeClassifier max_depth=15", "n_estimators": 100}
    )

CV: {'accuracy': {'mean': 0.8649303311728461, 'std': 0.004584899140872233}, 'balanced_accuracy': {'mean': 0.7830769550482949, 'std': 0.005618434415946675}, 'precision': {'mean': 0.7707452683656615, 'std': 0.013775916570700346}, 'recall': {'mean': 0.6251749456952954, 'std': 0.008229154365584466}, 'f1': {'mean': 0.6903444753346509, 'std': 0.009793403634870527}, 'roc_auc': {'mean': 0.9163178164421503, 'std': 0.0023264230464021193}, 'average_precision': {'mean': 0.8070483106968263, 'std': 0.005789189593483983}}
Test: {'accuracy': 0.8647503224617652, 'balanced_accuracy': 0.7810756338576823, 'precision': 0.7614503816793893, 'recall': 0.6224648985959438, 'f1': 0.6849785407725322, 'roc_auc': 0.9141445553278504, 'average_precision': 0.8012850305356569}
Confusion matrix: [[11685, 750], [1452, 2394]]
Saved to /Users/kang/Desktop/ML project/Estimating-Income/results/bagging_tree.json


In [4]:
# Random Forest

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
)

run_and_save(
    "random_forest",
    rf,
    extra = {"n_estimators": 100, "max_depth": 15}
)



CV: {'accuracy': {'mean': 0.8637326037775142, 'std': 0.0024664503008870525}, 'balanced_accuracy': {'mean': 0.7690085692893429, 'std': 0.0034786268673384553}, 'f1': {'mean': 0.6744846082384991, 'std': 0.005591167583956669}, 'roc_auc': {'mean': 0.9187682388019685, 'std': 0.0019404883421321206}, 'average_precision': {'mean': 0.8088518995979396, 'std': 0.00493120484634404}}
Test: {'accuracy': 0.8656102205024261, 'balanced_accuracy': 0.7685283285879083, 'f1': 0.6726511071214841, 'roc_auc': 0.9170252865603163, 'average_precision': 0.8019555146878137}
Confusion matrix: [[11845, 590], [1598, 2248]]
Saved to /Users/kang/Desktop/ML project/Estimating-Income/results/random_forest.json
